# 01 · Collect 5 credit agreements from EDGAR (Milestone 1)

Downloads the five exhibits listed in `configs/collect.yaml`, cleans them to text, and writes
`raw/`, `text/` and `manifest.csv` (url, issuer, filing date, agreement date, sha256, char
count) to the private HF dataset repo under `m1_collect/`.

**No GPU needed.** On Kaggle, set Accelerator to **None**, so this does not use GPU quota and
can run while another notebook is using the T4s. Internet must be **on**.
**Secrets:** `HF_TOKEN` (write) and `SEC_USER_AGENT`, your name plus a contact email, for
example `Jane Doe jane@example.com`, which SEC requires. Requests are limited to 5/s, and
SEC's limit is 10/s.
Resumable: documents already in the Hub with a matching sha256 are not downloaded again.

In [ ]:
REPO_URL = "https://github.com/z26qin/credit-extractor.git"
BRANCH = "m1-collect"

In [ ]:
import os, subprocess, sys, site
ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else (
    "/content" if os.path.exists("/content") else os.getcwd())
REPO = os.path.join(ROOT, "credit-extractor")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
with open(os.path.join(site.getsitepackages()[0], "credit_extractor.pth"), "w") as f:
    f.write(REPO + "\n")      # `import src` survives a kernel restart
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
# after a kernel restart, start again from this cell
from src.platform import setup
env = setup()
assert env.data_repo, "HF_TOKEN missing: attach it as a secret (outputs must go to the Hub)"
assert env.secrets["SEC_USER_AGENT"], "SEC_USER_AGENT missing: add a secret like 'Jane Doe jane@example.com'"

In [ ]:
from src import edgar
cfg = edgar.load_cfg()
rows = edgar.run(env, cfg)

In [ ]:
import pandas as pd
display(pd.DataFrame(edgar.summary(rows)))
root = env.paths.work / cfg["hub_prefix"]
for r in rows:
    t = (root / "text" / f"{r['id']}.txt").read_text()
    print(f"===== {r['id']}  {r['title']!r}  ({len(t):,} chars)\n{t[:700]}\n...")

In [ ]:
# manifest for the GitHub commit (copy this output into reports/m1_manifest.csv,
# or send it back so it can be committed)
print((root / "manifest.csv").read_text())